# COIN on TSP and Multi-objective TSP (up to 24 cities)
This library example uses deterministic teaching fixtures. A chromosome is a permutation of city IDs; evaluation closes the tour from the last city back to the first. All objectives are minimized.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from coin.core import PermutationCoinAlgorithm, MultiObjectiveCoinAlgorithm
from coin.models import EdgeConfig, OptimizedEdgeCoin
from coin.problems.tsp import get_tsp_instance, list_tsp_instances, TSPProblem
[(x.id, x.dimension, x.available_objective_names) for x in list_tsp_instances()]

## Single-objective TSP
Run Edge COIN on the preloaded 16-city distance instance and inspect best-so-far learning progress.

In [ ]:
instance = get_tsp_instance('tsp-16')
problem = TSPProblem(instance, ('distance',))
config = EdgeConfig(problem_size=problem.dimension, population_size=100, reward_ratio=10, punishment_ratio=10, training_rate=5, objective='min')
algorithm = PermutationCoinAlgorithm(OptimizedEdgeCoin(config, seed=1), problem)
algorithm.run(100)
best_so_far = np.minimum.accumulate([point.best for point in algorithm.history])
plt.plot([point.evaluations for point in algorithm.history], best_so_far)
plt.xlabel('objective evaluations'); plt.ylabel('best distance (lower is better)'); plt.grid(alpha=.25);

## Bi-objective MO-TSP
The two matrices represent geographic distance and operating cost. MO COIN learns by nondominated depth/diversity and retains a unique external Pareto archive.

In [ ]:
mo_instance = get_tsp_instance('motsp-16')
mo_problem = TSPProblem(mo_instance, ('distance', 'operating_cost'))
mo_config = EdgeConfig(problem_size=mo_problem.dimension, population_size=100, reward_ratio=10, punishment_ratio=10, training_rate=5, objective='min')
mo_algorithm = MultiObjectiveCoinAlgorithm(OptimizedEdgeCoin(mo_config, seed=1), mo_problem)
for _ in range(100): mo_algorithm.step()
front = mo_algorithm.archive_values
order = np.argsort(front[:, 0])
plt.scatter(front[:, 0], front[:, 1], label='nondominated tours')
plt.plot(front[order, 0], front[order, 1], alpha=.5)
plt.xlabel('distance'); plt.ylabel('operating cost'); plt.grid(alpha=.25); plt.legend();
print('archive size:', len(front), 'evaluations:', mo_algorithm.evaluations)

## Reproducible comparison checklist
Use the same `TSPProblem` object, population, generations/evaluation budget, and seeds for every algorithm. Report every seed, best/mean for scalar TSP, and convergence, spread, hypervolume or nondominated ratio for MO-TSP. The preloads are controlled teaching fixtures—not claims of published best-known benchmarks.